# CartoonStudio — Kaggle 3D story renderer

Edit the **Your story input** cell, then run the notebook cells to create an H.264 video with dialogue, lip-sync cues, acting cues, lighting, and a scene selected from your visual prompt. Enter dialogue line by line and choose a style (`auto`, `human`, or `cartoon`). The prompt guides the scene and look; it does not create detailed 3D models from text.

The bundled characters are procedural cartoon prototypes. For detailed characters, attach rigged GLB/GLTF assets or enable the optional Ara generator in Cell 4. Blender uses Cycles CUDA when available and falls back to EEVEE otherwise. The notebook reports imported skeletons, animation clips, and facial controls; it does not promise photorealistic output from its bundled assets.


In [ ]:
# Extract the project source embedded in this notebook (no Kaggle input dataset required).
from pathlib import Path
import base64, hashlib, zipfile, io, subprocess, sys

ROOT = Path('/kaggle/working/CartoonStudio')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = '...'
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
expected_sha256 = '3d6840578d15ec17c271cb394eab8f2d6c88549a8dfc1eaad1e68380d86e4dfb'
if hashlib.sha256(archive_bytes).hexdigest() != expected_sha256:
    raise RuntimeError('Embedded CartoonStudio source archive checksum mismatch.')
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for name in z.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError(f'Unsafe path in embedded project archive: {name}')
    z.extractall('/kaggle/working')
print('Embedded source extracted:', ROOT)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'No NVIDIA GPU detected')


In [ ]:
# Install Blender, FFmpeg, Piper and a compatible Node runtime for Ara 3D Gen.
import subprocess, sys, shutil, os, platform, tarfile, io, requests
base_tools = ('ffmpeg', 'ffprobe', 'espeak', 'git', 'node', 'npm')
if not shutil.which('blender') or any(not shutil.which(tool) for tool in base_tools):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'blender', 'ffmpeg', 'espeak', 'git', 'nodejs', 'npm', 'xz-utils'], check=True)
if not shutil.which('blender'):
    raise RuntimeError('Blender installation failed: executable not found after apt install.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'piper-tts', 'requests'], check=True)
node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    # Kaggle images may ship Node 18. Install the current Node 22 LTS binary into
    # the session and put it first on PATH so npm uses it too.
    arch = {'x86_64':'x64', 'aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise RuntimeError(f'Unsupported Kaggle CPU architecture for Node: {platform.machine()}')
    releases = requests.get('https://nodejs.org/dist/index.json', timeout=30)
    releases.raise_for_status()
    release = next(item for item in releases.json() if item['version'].startswith('v22.') and f'linux-{arch}' in item['files'])
    version = release['version']
    archive_url = f'https://nodejs.org/dist/{version}/node-{version}-linux-{arch}.tar.xz'
    response = requests.get(archive_url, timeout=180)
    response.raise_for_status()
    node_root = Path('/kaggle/working/node22')
    node_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(response.content), mode='r:xz') as tf:
        base = f'node-{version}-linux-{arch}/'
        members = [m for m in tf.getmembers() if m.name.startswith(base)]
        for member in members:
            relative = Path(member.name[len(base):])
            target = (node_root / relative).resolve()
            if not target.is_relative_to(node_root.resolve()):
                raise RuntimeError('Unsafe path in Node.js release archive')
        tf.extractall(node_root, members=members, filter='data')
    os.environ['PATH'] = str(node_root / f'node-{version}-linux-{arch}' / 'bin') + os.pathsep + os.environ['PATH']
    node_major = int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major < 20:
    raise RuntimeError(f'Node 20+ is required by Ara 3D Gen; found {node_major}.')
print('Blender:', subprocess.check_output(['blender','--version'],text=True).splitlines()[0])
print('FFmpeg:', subprocess.check_output(['ffmpeg','-version'],text=True).splitlines()[0])
print('Node:', subprocess.check_output(['node','--version'],text=True).strip())


In [ ]:
# Select Cycles only when this Blender build can enumerate a CUDA device; Kaggle may expose an unusable CUDA runtime.
probe_expr="import bpy; a=bpy.context.preferences.addons.get('cycles'); p=a.preferences if a else None; p.compute_device_type='CUDA' if p else None; p.get_devices() if p else None; d=[x.name for x in p.devices if x.type=='CUDA'] if p else []; print('CYCLES_CUDA_DEVICES', d)"
RENDERER='eevee'; STRICT_GPU=False
try:
    probe=subprocess.run(['blender','-b','--python-expr',probe_expr],capture_output=True,text=True,timeout=35)
    print(probe.stdout[-1500:]); print(probe.stderr[-800:])
    if probe.returncode==0 and "CYCLES_CUDA_DEVICES ['" in probe.stdout:
        RENDERER='cycles'; STRICT_GPU=True
except subprocess.TimeoutExpired:
    print('Blender CUDA discovery timed out; using EEVEE.')
print('Selected renderer:',RENDERER,'strict GPU:',STRICT_GPU)


In [ ]:
# Prepare speech tools and optional character sources. Ara asset generation is opt-in because its full build can take many minutes.
import json, os, requests, zipfile
from pathlib import Path
ARA_ROOT=Path('/kaggle/working/ara-3d-gen')
ARA_READY=(ARA_ROOT/'public/assets/catalog.json').is_file()
ENABLE_ARA=False  # Set True only when you want to spend time building Ara's character catalog.
if ENABLE_ARA and not ARA_READY:
    try:
        if not (ARA_ROOT/'package.json').exists():
            subprocess.run(['git','clone','--depth','1','https://github.com/olatunde85/3d-gen.git',str(ARA_ROOT)],check=True,timeout=300)
        if not (ARA_ROOT/'node_modules').exists():
            subprocess.run(['npm','ci'],cwd=ARA_ROOT,check=True,timeout=600)
        vendor=ARA_ROOT/'vendor'
        task='build-assets' if (vendor/'mh/makehuman/data').is_dir() and (vendor/'sys/skins').is_dir() else 'setup-assets'
        print('Ara asset task:',task,'(maximum 12 minutes)',flush=True)
        subprocess.run(['npm','run',task],cwd=ARA_ROOT,check=True,timeout=720)
        ARA_READY=(ARA_ROOT/'public/assets/catalog.json').is_file()
    except (subprocess.TimeoutExpired,subprocess.CalledProcessError) as exc:
        print('Ara setup unavailable:',repr(exc),flush=True)
RHU_DIR=Path('/kaggle/working/rhubarb'); RHU_DIR.mkdir(parents=True,exist_ok=True)
rhubarb_binary=next(RHU_DIR.rglob('rhubarb'),None)
if rhubarb_binary is None:
    release=requests.get('https://api.github.com/repos/DanielSWolf/rhubarb-lip-sync/releases/latest',timeout=30); release.raise_for_status()
    asset=next((a for a in release.json()['assets'] if 'Linux.zip' in a['name']),None)
    if asset is None: raise RuntimeError('Rhubarb Linux release asset is unavailable.')
    archive=RHU_DIR/'rhubarb-linux.zip'; response=requests.get(asset['browser_download_url'],timeout=120); response.raise_for_status(); archive.write_bytes(response.content)
    with zipfile.ZipFile(archive) as zf:
        for name in zf.namelist():
            if Path(name).is_absolute() or '..' in Path(name).parts: raise ValueError(f'Unsafe Rhubarb archive path: {name}')
        zf.extractall(RHU_DIR)
rhubarb_binary=next(RHU_DIR.rglob('rhubarb'),None)
if rhubarb_binary is None: raise FileNotFoundError('Rhubarb binary missing after extraction.')
rhubarb_binary.chmod(0o755)
# For direct assets, create /kaggle/working/character_files.json, e.g. {"john":"/kaggle/input/my-assets/john.glb"}.
CHARACTER_FILES=Path('/kaggle/working/character_files.json')
print('Ara catalog ready:',ARA_READY,'; enabled:',ENABLE_ARA); print('Phoneme lip-sync ready:',rhubarb_binary)
print('Direct rigged assets configured:',CHARACTER_FILES.is_file())


In [ ]:
# Download one Piper voice directly from the public voice repository.
voices=ROOT/'voices'; voices.mkdir(exist_ok=True)
base='https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_GB/alan/medium'
for filename in ('en_GB-alan-medium.onnx','en_GB-alan-medium.onnx.json'):
    path=voices/filename
    if not path.exists():
        response=requests.get(f'{base}/{filename}',timeout=180); response.raise_for_status(); path.write_bytes(response.content)
print('Voice model ready:',[(p.name,round(p.stat().st_size/1e6,1)) for p in voices.iterdir()])


In [ ]:
# Paste your scene prompt and dialogue here. This is the story the render cell uses.
# The visual prompt controls the scene/style description; dialogue entries control speech and acting cues.
STORY_TITLE = "Your story title"
VISUAL_PROMPT = "Describe the characters, place, lighting, mood, and camera framing. Example: Two friends meet in a sunlit forest clearing at golden hour; natural adult humans, warm cinematic light, medium two-shot."
VISUAL_STYLE = "auto"  # auto, human, or cartoon
CHARACTERS = {
    "alex": {"voice": "en_GB-alan-medium", "color": "blue"},
    "sam": {"voice": "en_GB-alan-medium", "color": "coral"},
}
DIALOGUE = [
    {"speaker": "alex", "text": "Paste the first line of dialogue.", "emotion": "happy", "action": "wave"},
    {"speaker": "sam", "text": "Paste the reply.", "emotion": "surprised", "action": "nod"},
]
# Add dialogue entries for each spoken line. Speaker must match a key in CHARACTERS.
# Supported emotions: neutral, happy, surprised, sad, angry, excited. Actions: none, wave, nod, point.
from pathlib import Path
import json
USER_STORY_PATH = ROOT / 'stories' / 'user_story.json'
USER_STORY_PATH.parent.mkdir(parents=True, exist_ok=True)
USER_STORY_PATH.write_text(json.dumps({
    "title": STORY_TITLE, "scene": "auto", "visual_style": VISUAL_STYLE,
    "visual_prompt": VISUAL_PROMPT, "characters": CHARACTERS, "dialogue": DIALOGUE,
}, indent=2), encoding='utf-8')
print('Story saved for rendering:', USER_STORY_PATH)
print('Dialogue lines:', len(DIALOGUE), '| visual style:', VISUAL_STYLE)


In [ ]:
# Render your story at 720p/24. Change --preview-seconds 3 to 0 for the full story.
import time
OUT=Path('/kaggle/working/cartoon_studio_output'); OUT.mkdir(exist_ok=True)
cmd=[sys.executable,'-m','cartoon_studio.cli','generate','--story',str(USER_STORY_PATH),'--voices',str(voices),'--work',str(OUT/'work'),'--output',str(OUT/'first_cartoon.mp4'),'--style','auto','--quality','high','--rhubarb',str(rhubarb_binary),'--tts','piper','--renderer',RENDERER,'--width','1280','--height','720','--fps','24','--samples','32','--preview-seconds','3']
if CHARACTER_FILES.is_file(): cmd += ['--character-files',str(CHARACTER_FILES)]
elif ARA_READY: cmd += ['--ara-root',str(ARA_ROOT)]
else: print('No detailed rigged character source configured. Output will use procedural prototypes; supply rigged assets for human-quality characters.')
if STRICT_GPU: cmd.append('--strict-gpu')
t0=time.monotonic(); subprocess.run(cmd,check=True,cwd=ROOT)
print('Video output:',OUT/'first_cartoon.mp4'); print('Total elapsed seconds:',round(time.monotonic()-t0,1))
from IPython.display import display,Video
display(Video(str(OUT/'first_cartoon.mp4'),embed=True))


In [ ]:
# Estimate render time from the measured frames when a render log is available.
import json
stats_path=OUT/'work'/'render_log.json'
if stats_path.exists():
    stats=json.loads(stats_path.read_text()); spf=stats['seconds_per_frame']
    subprocess.run([sys.executable,'-m','cartoon_studio.budget','--minutes','20','--seconds-per-frame',str(spf),'--weekly-hours','30','--reserve','0.15'],cwd=ROOT,check=True)
    print('Measured seconds per frame including Blender launch:',round(spf,3))
else: print('Render reused cached chunks; no fresh render timing was recorded.')


## Story and quality settings

Edit the **Your story input** cell before running the render cell. Put your visual direction in `VISUAL_PROMPT`, name the characters in `CHARACTERS`, and paste each spoken line into `DIALOGUE`. Add one dialogue entry per line; the speaker must match a character key. Supported acting cues are listed in the cell. `VISUAL_STYLE` accepts `auto`, `human`, or `cartoon`. The visual prompt selects a supported stage from its wording and informs the character style; it does not generate a detailed model, motion capture, or custom environment from text.

The render cell defaults to a 3-second 1280×720 preview. Change `--preview-seconds 3` to `--preview-seconds 0` for the full story. A ten-minute safety cap and resumable 20-second chunks are built into the CLI.

For detailed characters, attach freely licensed rigged `.glb` or `.gltf` files as Kaggle input and create `/kaggle/working/character_files.json` mapping character keys to their files. Assets should include body animation actions and facial blend shapes for expression and lip-sync. Cell 4 supports optional Ara generation; set `ENABLE_ARA=True` to download/build its character catalog (up to 12 minutes). Direct GLB/GLTF assets take priority. The default stage is a lightweight procedural set rather than a scanned cinematic environment.
